In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from scipy.special import i0
from scipy.signal import freqz
from IPython.display import display, HTML, Math
from ipywidgets import Output, HBox, Layout

# ============================================================
# PROBLEM 12.12.3 — KAISER-WINDOW LOW-PASS FIR DESIGN
# ============================================================

plt.rcParams.update({'font.size':11.5,'axes.titlesize':13.0,'axes.labelsize':11.5,'xtick.labelsize':10.5,'ytick.labelsize':10.5,'legend.fontsize':10.0})

# ============================================================
# CSS
# ============================================================

display(HTML("""
<style>
.ex-root{width:970px;max-width:970px;font-family:Arial,sans-serif;}
.ex-header{background:linear-gradient(90deg,#7b1fa2,#ab47bc);color:white;padding:10px 15px;border-radius:8px 8px 0 0;font-size:18px;font-weight:bold;}
.ex-intro{background:#fbf7fc;border:1px solid #d7c4e2;border-top:none;padding:10px 13px;border-radius:0 0 8px 8px;font-size:13.5px;line-height:1.5;margin-bottom:10px;}
.ex-box{width:944px;border:1px solid #d7c4e2;border-radius:7px;padding:10px 12px;margin-bottom:10px;font-size:13.5px;line-height:1.55;}
.ex-title{font-weight:bold;color:#6a1b9a;font-size:14px;margin-bottom:7px;}
.ex-value{font-weight:bold;}
.ex-columns{display:flex;gap:40px;align-items:flex-start;}
.ex-column{flex:1;min-width:0;}
</style>
"""))

# ============================================================
# INTRODUCTION
# ============================================================

display(HTML("""
<div class="ex-root">
<div class="ex-header">Problem 12.12.3 — Kaiser-Window Low-Pass FIR Filter</div>
<div class="ex-intro">
The Kaiser-window design equations are evaluated numerically to determine α and the FIR filter order N. The design is then completed by constructing the delayed ideal low-pass impulse response, applying the Kaiser window, and plotting the resulting magnitude response and FIR impulse response.
</div>
</div>
"""))

# ============================================================
# GIVEN SPECIFICATIONS
# ============================================================

wp_n, ws_n, wc_n = 0.63, 0.65, 0.64
wp, ws, wc = wp_n*np.pi, ws_n*np.pi, wc_n*np.pi
delta_p, delta_s = 0.02, 0.15
delta = min(delta_p,delta_s)
Omega_s = 2*np.pi

display(HTML(f"""
<div class="ex-box">
<div class="ex-title">Given specifications</div>
<div class="ex-columns">

<div class="ex-column">
Passband: <span class="ex-value">0.98 &lt; |H| &lt; 1.02</span><br>
Frequency range: <span class="ex-value">0 ≤ |ω| ≤ {wp_n:.2f}π</span><br>
Passband deviation: <span class="ex-value">δp = {delta_p:.2f}</span>
</div>

<div class="ex-column">
Stopband: <span class="ex-value">|H| &lt; {delta_s:.2f}</span><br>
Frequency range: <span class="ex-value">{ws_n:.2f}π ≤ |ω| ≤ π</span><br>
Ideal cutoff: <span class="ex-value">ωc = {wc_n:.2f}π</span>
</div>

</div>
</div>
"""))

# ============================================================
# NUMERICAL KAISER DESIGN
# ============================================================

As = -20*np.log10(delta)

if As <= 21:
    alpha = 0.0
elif As <= 50:
    alpha = 0.5842*(As-21)**0.4 + 0.07886*(As-21)
else:
    alpha = 0.1102*(As-8.7)

dw = ws-wp
dw_n = dw/np.pi
D = 0.9222 if As <= 21 else (As-7.95)/14.36
order_argument = Omega_s*D/dw
N = 1+int(order_argument)
delay = N/2

# ============================================================
# NUMERICAL EQUATIONS — TWO COLUMNS
# ============================================================

left_output = Output(layout=Layout(width='48%'))
right_output = Output(layout=Layout(width='48%'))

with left_output:
    display(Math(rf"\delta_p=1.02-1={delta_p:.2f},\qquad \delta_s={delta_s:.2f}"))
    display(Math(rf"\delta=\min(\delta_p,\delta_s)=\min({delta_p:.2f},{delta_s:.2f})={delta:.2f}"))
    display(Math(rf"A_s=-20\log_{{10}}(\delta)=-20\log_{{10}}({delta:.2f})={As:.6f}\,\mathrm{{dB}}"))
    display(Math(rf"\alpha=0.5842(A_s-21)^{{0.4}}+0.07886(A_s-21)={alpha:.6f}"))

with right_output:
    display(Math(rf"\Delta\omega=\omega_s-\omega_p={ws_n:.2f}\pi-{wp_n:.2f}\pi={dw_n:.2f}\pi"))
    display(Math(rf"D=\frac{{A_s-7.95}}{{14.36}}=\frac{{{As:.6f}-7.95}}{{14.36}}={D:.6f}"))
    display(Math(rf"N\geq1+\left[\frac{{2\pi D}}{{\Delta\omega}}\right]=1+[\,{order_argument:.4f}\,]={N}"))
    display(Math(rf"\tau=\frac{{N}}{{2}}=\frac{{{N}}}{{2}}={delay:.1f}\,\mathrm{{samples}}"))

display(HBox([left_output,right_output],layout=Layout(width='970px',justify_content='space-between',align_items='flex-start')))

# ============================================================
# NUMERICAL SUMMARY
# ============================================================

display(HTML(f"""
<div class="ex-box">
<div class="ex-title">Numerical design result</div>
<div class="ex-columns">

<div class="ex-column">
δ<sub>p</sub> = <span class="ex-value">{delta_p:.2f}</span><br>
δ<sub>s</sub> = <span class="ex-value">{delta_s:.2f}</span><br>
Governing δ = <span class="ex-value">{delta:.2f}</span><br>
A<sub>s</sub> = <span class="ex-value">{As:.6f} dB</span><br>
α = <span class="ex-value">{alpha:.6f}</span>
</div>

<div class="ex-column">
Δω = <span class="ex-value">{dw_n:.3f}π</span><br>
ω<sub>c</sub> = <span class="ex-value">{wc_n:.3f}π</span><br>
D = <span class="ex-value">{D:.6f}</span><br>
Filter order N = <span class="ex-value">{N}</span><br>
Filter length N+1 = <span class="ex-value">{N+1}</span><br>
Delay = <span class="ex-value">{delay:.1f} samples</span>
</div>

</div>
</div>
"""))

# ============================================================
# IDEAL LOW-PASS IMPULSE RESPONSE
# ============================================================

n = np.arange(N+1,dtype=float)
m = n-delay
hd = np.empty_like(m)
center = np.abs(m)<1e-12
hd[center] = wc/np.pi
hd[~center] = np.sin(wc*m[~center])/(np.pi*m[~center])

display(HTML("""
<div class="ex-box">
<div class="ex-title">Ideal low-pass impulse response</div>
The ideal low-pass response with cutoff frequency ωc, shifted by the generalized-linear-phase delay, is
</div>
"""))

display(Math(rf"h_d[n]=\frac{{\sin[{wc_n:.3f}\pi(n-{delay:.1f})]}}{{\pi(n-{delay:.1f})}}"))

# ============================================================
# KAISER WINDOW AND ACTUAL FIR FILTER
# ============================================================

x = (n-delay)/delay
w = i0(alpha*np.sqrt(np.maximum(0,1-x**2)))/i0(alpha)
h = hd*w

# ============================================================
# FREQUENCY RESPONSE
# ============================================================

omega,H = freqz(h,worN=8192)
omega_n = omega/np.pi
Hmag = np.abs(H)

# ============================================================
# NUMERICAL VERIFICATION
# ============================================================

pass_mask = omega_n <= wp_n
stop_mask = omega_n >= ws_n

pass_min = np.min(Hmag[pass_mask])
pass_max = np.max(Hmag[pass_mask])
stop_max = np.max(Hmag[stop_mask])

display(HTML(f"""
<div class="ex-box">
<div class="ex-title">Numerical verification of the resulting FIR filter</div>
<div class="ex-columns">

<div class="ex-column">
Minimum magnitude in passband: <span class="ex-value">{pass_min:.6f}</span><br>
Maximum magnitude in passband: <span class="ex-value">{pass_max:.6f}</span>
</div>

<div class="ex-column">
Maximum magnitude in stopband: <span class="ex-value">{stop_max:.6f}</span><br>
Calculated order: <span class="ex-value">N = {N}</span>
</div>

</div>
</div>
"""))

# ============================================================
# FINAL FIGURES
# ============================================================

fig,(ax1,ax2) = plt.subplots(1,2,figsize=(13.5,4.6))

# ============================================================
# 1. MAGNITUDE RESPONSE
# ============================================================

ax1.plot(omega_n,Hmag,color='red',linewidth=1.7,label='Actual FIR response')

# Ideal response
ax1.plot([0,wc_n],[1,1],'--',linewidth=1.2,label='Ideal response')
ax1.plot([wc_n,wc_n],[1,0],'--',linewidth=1.2)
ax1.plot([wc_n,1],[0,0],'--',linewidth=1.2)

# Passband limits
ax1.plot([0,wp_n],[0.98,0.98],':',linewidth=1.1,label='Specification limits')
ax1.plot([0,wp_n],[1.02,1.02],':',linewidth=1.1)

# Stopband limit
ax1.plot([ws_n,1],[0.15,0.15],':',linewidth=1.1)

# Frequency edges and cutoff
ax1.axvline(wp_n,linestyle=':',linewidth=1.0)
ax1.axvline(ws_n,linestyle=':',linewidth=1.0)
ax1.axvline(wc_n,linestyle='-.',linewidth=1.0,label=r'Ideal cutoff $\omega_c$')

# Transition region
ax1.axvspan(wp_n,ws_n,alpha=0.08)

ax1.set_xlim(0,1)
ax1.set_ylim(-0.05,1.10)
ax1.set_title('Magnitude Response and Design Specifications')
ax1.set_xlabel(r'Normalized frequency $\omega/\pi$')
ax1.set_ylabel(r'$|H(e^{j\omega})|$')
ax1.grid(True,linestyle=':',alpha=0.25)
ax1.legend(loc='upper center',bbox_to_anchor=(0.5,-0.16),ncol=3,frameon=False)

# ============================================================
# 2. IMPULSE RESPONSE
# ============================================================

markerline,stemlines,baseline = ax2.stem(n,h,linefmt='r-',markerfmt='ro',basefmt=' ')
plt.setp(markerline,markersize=3.5)
plt.setp(stemlines,linewidth=1.0)

ax2.axvline(delay,linestyle='--',linewidth=1.1,label=rf'Delay = {delay:.1f}')
ax2.set_title('Kaiser-Window FIR Impulse Response')
ax2.set_xlabel('Sample index $n$')
ax2.set_ylabel('$h[n]$')
ax2.grid(True,linestyle=':',alpha=0.25)
ax2.legend(loc='upper center',bbox_to_anchor=(0.5,-0.16),frameon=False)

plt.subplots_adjust(left=0.07,right=0.98,top=0.90,bottom=0.23,wspace=0.30)
plt.show()
plt.close(fig)